# MVP de Engenharia de Dados — Produção de Café no Brasil

## 02 — Transformação e Camada Silver

### Objetivo

Este notebook tem como objetivo realizar o processamento e a transformação dos dados armazenados na camada Bronze, convertendo os dados brutos provenientes do SIDRA/IBGE em uma estrutura organizada e adequada para análise.

Na camada Silver serão realizadas etapas de interpretação da estrutura do arquivo, limpeza, padronização, tratamento dos valores especiais, conversão dos tipos de dados e reorganização das informações.

Ao final desta etapa, espera-se obter uma tabela estruturada e confiável, preservando as informações necessárias para responder às perguntas de negócio definidas para o MVP.

### Origem dos dados

Os dados utilizados nesta etapa são provenientes da tabela Delta da camada Bronze:

`workspace.bronze.ibge_pam_cafe_raw`

A tabela Bronze contém 204 registros e preserva cada linha do arquivo original, além do metadado técnico referente à data de ingestão.

In [0]:
%sql

SELECT
    linha_original
FROM workspace.bronze.ibge_pam_cafe_raw;

### Identificação dos blocos de variáveis

A análise exploratória identificou cinco blocos de informações no arquivo de origem:

- Área destinada à colheita;
- Área colhida;
- Quantidade produzida;
- Rendimento médio da produção;
- Valor da produção.

Como o arquivo exportado pelo SIDRA apresenta essas variáveis em blocos sucessivos, será necessário identificar a posição de cada bloco antes da transformação para o formato estruturado da camada Silver.

In [0]:
caminho_arquivo = "/Volumes/workspace/bronze/dados_brutos/ibge_pam_cafe_2015_2025.csv"

with open(caminho_arquivo, "r", encoding="utf-8-sig") as arquivo:
    linhas = arquivo.readlines()

for numero, linha in enumerate(linhas, start=1):
    if "Variável -" in linha:
        print(numero, linha.strip())

In [0]:
#exibindo o primeiro bloco para saber quais linhas são cabeçalho
for numero in range(1, 38):
    print(f"{numero}: {linhas[numero-1].strip()}")

In [0]:
#verificando se existem linhas de rodapé, fonte/notas ou separadores depois dos estados.
for numero in range(30, 38):
    print(f"{numero}: {linhas[numero-1].strip()}")

In [0]:
#verificando onde cada bloco termina
for numero, linha in enumerate(linhas, start=1):
    if "Fonte:" in linha:
        print(numero, linha.strip())

### Análise da estrutura do arquivo de origem

Antes da transformação dos dados, foi realizada uma análise da estrutura do arquivo CSV obtido do SIDRA/IBGE.

Foi identificado que o arquivo não possui uma estrutura tabular convencional com um único cabeçalho. Os dados estão organizados em cinco blocos independentes, correspondentes às seguintes variáveis:

1. Área destinada à colheita (Hectares)
2. Área colhida (Hectares)
3. Quantidade produzida (Toneladas)
4. Rendimento médio da produção (Quilogramas por Hectare)
5. Valor da produção (Mil Reais)

Cada bloco contém os anos de 2015 a 2025, os produtos relacionados à produção de café e os registros das Unidades da Federação.

Também foi verificado que cada bloco termina com a indicação da fonte "IBGE - Produção Agrícola Municipal".

Essa análise da estrutura do arquivo é necessária para orientar o processo de transformação e padronização dos dados que será realizado na camada Silver.

In [0]:
#cria schema silver
%sql

CREATE SCHEMA IF NOT EXISTS workspace.silver
COMMENT 'Camada Silver do MVP de Engenharia de Dados - dados tratados, padronizados e estruturados para análise';

### Transformação e estruturação dos dados

Após a análise do arquivo de origem, inicia-se o processo de transformação dos dados para a camada Silver.

Os cinco blocos identificados no arquivo serão tratados e reorganizados em uma estrutura tabular única. Durante esse processo, os dados referentes às Unidades da Federação, anos, produtos e variáveis da produção agrícola serão estruturados de forma adequada para consultas e análises posteriores.

Também serão tratados os valores ausentes representados por símbolos no arquivo original e os tipos de dados serão padronizados quando necessário.

O objetivo desta etapa é transformar os dados brutos preservados na camada Bronze em dados estruturados e preparados para análise, mantendo a rastreabilidade em relação à fonte original.

In [0]:
# Verificação das linhas finais do arquivo: notas e legenda

for numero in range(170, len(linhas) + 1):
    print(f"{numero}: {linhas[numero-1].strip()}")

In [0]:
# Verificação do final das notas e da legenda(linha 190 em diante)

for numero in range(190, len(linhas) + 1):
    print(f"{numero}: {linhas[numero-1].strip()}")

In [0]:
# Identificação da transição entre Notas e Legenda

for numero in range(184, 191):
    print(f"{numero}: {linhas[numero-1].strip()}")

## Estrutura identificada no arquivo de origem

A inspeção do arquivo CSV do IBGE permitiu identificar que os dados estão organizados em cinco blocos, correspondentes às seguintes variáveis:

1. Área destinada à colheita (Hectares)
2. Área colhida (Hectares)
3. Quantidade produzida (Toneladas)
4. Rendimento médio da produção (Quilogramas por Hectare)
5. Valor da produção (Mil Reais)

Cada bloco possui informações referentes às Unidades da Federação, aos anos de 2015 a 2025 e aos produtos Café (em grão) Total, Café (em grão) Arábica e Café (em grão) Canephora.

Também foram identificadas, ao final do arquivo, as seções de **Notas** e **Legenda**, fornecidas pelo IBGE para explicar particularidades metodológicas e o significado dos símbolos utilizados nos dados.

Essas seções possuem caráter descritivo e não representam registros da produção agrícola. Por esse motivo, serão preservadas no arquivo original da camada Bronze, mas não serão incorporadas à tabela analítica da camada Silver.

Na camada Silver, os cinco blocos serão extraídos, tratados e integrados em uma estrutura tabular padronizada, adequada às etapas posteriores de análise.

In [0]:
# Extração do primeiro bloco:
# Área destinada à colheita (Hectares)

bloco_area_destinada = linhas[1:34]

print("Quantidade de linhas extraídas:", len(bloco_area_destinada))

for numero, linha in enumerate(bloco_area_destinada, start=2):
    print(f"{numero}: {linha.strip()}")

In [0]:
# Separação apenas das linhas de dados das UFs
# Primeiro bloco: Área destinada à colheita

dados_area_destinada = linhas[5:33]

print("Quantidade de UFs encontradas:", len(dados_area_destinada))

for linha in dados_area_destinada:
    print(linha.strip())

In [0]:
import csv

# Verifica a quantidade de campos de cada registro do primeiro bloco
for linha in dados_area_destinada:
    campos = next(csv.reader([linha]))
    print(campos[0], "->", len(campos), "campos")

In [0]:
# Leitura das três linhas que formam o cabeçalho do primeiro bloco

cabecalho_nivel1 = next(csv.reader([linhas[2]]))
cabecalho_anos   = next(csv.reader([linhas[3]]))
cabecalho_produtos = next(csv.reader([linhas[4]]))

print("Campos nível 1:", len(cabecalho_nivel1))
print("Campos anos:", len(cabecalho_anos))
print("Campos produtos:", len(cabecalho_produtos))

print("\n--- ANOS ---")
print(cabecalho_anos)

print("\n--- PRODUTOS ---")
print(cabecalho_produtos)

In [0]:
anos = list(range(2015, 2026))

produtos = [
    "Total",
    "Café (em grão) Total",
    "Café (em grão) Arábica",
    "Café (em grão) Canephora"
]

estrutura_colunas = []

for ano in anos:
    for produto in produtos:
        estrutura_colunas.append((ano, produto))

print("Quantidade de combinações:", len(estrutura_colunas))

for posicao, (ano, produto) in enumerate(estrutura_colunas, start=1):
    print(posicao, "->", ano, "|", produto)

In [0]:
# Teste de transformação com apenas uma UF: Rondônia

campos_rondonia = next(csv.reader([dados_area_destinada[0]]))

uf = campos_rondonia[0]
valores = campos_rondonia[1:]

registros_teste = []

for (ano, produto), valor in zip(estrutura_colunas, valores):
    registros_teste.append({
        "uf": uf,
        "ano": ano,
        "produto": produto,
        "valor": valor
    })

print("UF:", uf)
print("Quantidade de registros gerados:", len(registros_teste))

for registro in registros_teste:
    print(registro)

In [0]:
%whos

In [0]:
# Transformação de todas as UFs para o formato estruturado

registros_area_destinada = []

for linha in dados_area_destinada:
    campos = next(csv.reader([linha]))

    uf = campos[0]

    # Os 44 valores vêm depois da UF
    valores = campos[1:]

    for (ano, produto), valor in zip(estrutura_colunas, valores):
        registros_area_destinada.append({
            "uf": uf,
            "ano": ano,
            "produto": produto,
            "valor": valor
        })

print("Quantidade de UFs processadas:", len(dados_area_destinada))
print("Quantidade total de registros gerados:", len(registros_area_destinada))

# Exibir apenas os 10 primeiros para conferência
for registro in registros_area_destinada[:10]:
    print(registro)

In [0]:
# Conferência da primeira e da última UF processada

print("Primeiro registro:")
print(registros_area_destinada[0])

print("\nÚltimo registro:")
print(registros_area_destinada[-1])

print("\nQuantidade de registros por UF:")

from collections import Counter

contagem_ufs = Counter(
    registro["uf"] for registro in registros_area_destinada
)

print(contagem_ufs)

In [0]:
# Isolando o bloco: Área colhida (Hectares)
# Linhas 37 a 69 do arquivo original

bloco_area_colhida = linhas[36:69]

print("Quantidade de linhas extraídas:", len(bloco_area_colhida))

# Conferência do início e do final do bloco
print("\nPrimeira linha:")
print(bloco_area_colhida[0].strip())

print("\nÚltima linha:")
print(bloco_area_colhida[-1].strip())

In [0]:
# Extraindo somente as linhas das UFs do bloco Área colhida
# As 5 primeiras linhas são cabeçalhos
# A última linha é a fonte

dados_area_colhida = bloco_area_colhida[4:32]

print("Quantidade de UFs encontradas:", len(dados_area_colhida))

# Conferência da primeira e da última UF
print("\nPrimeira UF:")
print(dados_area_colhida[0].strip())

print("\nÚltima UF:")
print(dados_area_colhida[-1].strip())

In [0]:
# Transformação do bloco: Área colhida (Hectares)

registros_area_colhida = []

for linha in dados_area_colhida:
    campos = next(csv.reader([linha]))

    uf = campos[0]
    valores = campos[1:]

    for (ano, produto), valor in zip(estrutura_colunas, valores):
        registros_area_colhida.append({
            "uf": uf,
            "ano": ano,
            "produto": produto,
            "valor": valor
        })

print("Quantidade de UFs processadas:", len(dados_area_colhida))
print("Quantidade total de registros gerados:", len(registros_area_colhida))

# Exibir apenas os 10 primeiros para conferência
for registro in registros_area_colhida[:10]:
    print(registro)

In [0]:
# Validação da transformação: Área colhida

from collections import Counter

contagem_area_colhida = Counter(
    registro["uf"] for registro in registros_area_colhida
)

print("Primeiro registro:")
print(registros_area_colhida[0])

print("\nÚltimo registro:")
print(registros_area_colhida[-1])

print("\nQuantidade de registros por UF:")
print(contagem_area_colhida)

print("\nTodas as UFs possuem 44 registros?",
      all(qtd == 44 for qtd in contagem_area_colhida.values()))

In [0]:
# Localizando o bloco: Quantidade produzida (Toneladas)

inicio_quantidade = None
fim_quantidade = None

for i, linha in enumerate(linhas):
    if "Variável - Quantidade produzida (Toneladas)" in linha:
        inicio_quantidade = i

        # Procura a próxima linha Fonte
        for j in range(i, len(linhas)):
            if "Fonte:" in linhas[j]:
                fim_quantidade = j
                break

        break

print("Linha inicial:", inicio_quantidade + 1)
print("Linha final:", fim_quantidade + 1)

bloco_quantidade = linhas[inicio_quantidade:fim_quantidade + 1]

print("Quantidade de linhas extraídas:", len(bloco_quantidade))
print("\nPrimeira linha:")
print(bloco_quantidade[0].strip())

print("\nÚltima linha:")
print(bloco_quantidade[-1].strip())

In [0]:
# Conferindo o início do bloco Quantidade produzida

for i, linha in enumerate(bloco_quantidade[:8]):
    print(i, "->", linha.strip())

In [0]:
# Extraindo somente as linhas das UFs do bloco Quantidade produzida
# As 4 primeiras linhas são cabeçalhos
# A última linha é a fonte

dados_quantidade = bloco_quantidade[4:32]

print("Quantidade de UFs encontradas:", len(dados_quantidade))

print("\nPrimeira UF:")
print(dados_quantidade[0].strip())

print("\nÚltima UF:")
print(dados_quantidade[-1].strip())

In [0]:
# Transformação do bloco: Quantidade produzida (Toneladas)

registros_quantidade = []

for linha in dados_quantidade:
    campos = next(csv.reader([linha]))

    uf = campos[0]
    valores = campos[1:]

    for (ano, produto), valor in zip(estrutura_colunas, valores):
        registros_quantidade.append({
            "uf": uf,
            "ano": ano,
            "produto": produto,
            "valor": valor
        })

print("Quantidade de UFs processadas:", len(dados_quantidade))
print("Quantidade total de registros gerados:", len(registros_quantidade))

# Exibir apenas os 10 primeiros registros
for registro in registros_quantidade[:10]:
    print(registro)

In [0]:
# Validação da transformação: Quantidade produzida

contagem_quantidade = Counter(
    registro["uf"] for registro in registros_quantidade
)

print("Primeiro registro:")
print(registros_quantidade[0])

print("\nÚltimo registro:")
print(registros_quantidade[-1])

print("\nQuantidade de registros por UF:")
print(contagem_quantidade)

print("\nTodas as UFs possuem 44 registros?",
      all(qtd == 44 for qtd in contagem_quantidade.values()))

In [0]:
# Localizando o bloco: Rendimento médio da produção

inicio_rendimento = None
fim_rendimento = None

for i, linha in enumerate(linhas):
    if "Variável - Rendimento médio da produção (Quilogramas por Hectare)" in linha:
        inicio_rendimento = i

        # Procura a próxima linha Fonte
        for j in range(i, len(linhas)):
            if "Fonte:" in linhas[j]:
                fim_rendimento = j
                break

        break

print("Linha inicial:", inicio_rendimento + 1)
print("Linha final:", fim_rendimento + 1)

bloco_rendimento = linhas[inicio_rendimento:fim_rendimento + 1]

print("Quantidade de linhas extraídas:", len(bloco_rendimento))

print("\nPrimeira linha:")
print(bloco_rendimento[0].strip())

print("\nÚltima linha:")
print(bloco_rendimento[-1].strip())

In [0]:
# Conferindo o início do bloco Rendimento médio da produção

for i, linha in enumerate(bloco_rendimento[:8]):
    print(i, "->", linha.strip())

In [0]:
# Extraindo somente as linhas das UFs do bloco Rendimento médio

dados_rendimento = bloco_rendimento[4:32]

print("Quantidade de UFs encontradas:", len(dados_rendimento))

print("\nPrimeira UF:")
print(dados_rendimento[0].strip())

print("\nÚltima UF:")
print(dados_rendimento[-1].strip())

In [0]:
# Transformação do bloco: Rendimento médio da produção

registros_rendimento = []

for linha in dados_rendimento:
    campos = next(csv.reader([linha]))

    uf = campos[0]
    valores = campos[1:]

    for (ano, produto), valor in zip(estrutura_colunas, valores):
        registros_rendimento.append({
            "uf": uf,
            "ano": ano,
            "produto": produto,
            "valor": valor
        })

print("Quantidade de UFs processadas:", len(dados_rendimento))
print("Quantidade total de registros gerados:", len(registros_rendimento))

# Exibir apenas os 10 primeiros registros
for registro in registros_rendimento[:10]:
    print(registro)

In [0]:
# Validação da transformação: Rendimento médio da produção

contagem_rendimento = Counter(
    registro["uf"] for registro in registros_rendimento
)

print("Primeiro registro:")
print(registros_rendimento[0])

print("\nÚltimo registro:")
print(registros_rendimento[-1])

print("\nQuantidade de registros por UF:")
print(contagem_rendimento)

print("\nTodas as UFs possuem 44 registros?",
      all(qtd == 44 for qtd in contagem_rendimento.values()))

In [0]:
# Localizando o bloco: Valor da produção (Mil Reais)

inicio_valor_producao = None
fim_valor_producao = None

for i, linha in enumerate(linhas):
    if "Variável - Valor da produção (Mil Reais)" in linha:
        inicio_valor_producao = i

        # Procura a próxima linha Fonte
        for j in range(i, len(linhas)):
            if "Fonte:" in linhas[j]:
                fim_valor_producao = j
                break

        break

print("Linha inicial:", inicio_valor_producao + 1)
print("Linha final:", fim_valor_producao + 1)

bloco_valor_producao = linhas[inicio_valor_producao:fim_valor_producao + 1]

print("Quantidade de linhas extraídas:", len(bloco_valor_producao))

print("\nPrimeira linha:")
print(bloco_valor_producao[0].strip())

print("\nÚltima linha:")
print(bloco_valor_producao[-1].strip())

In [0]:
# Conferindo o início do bloco Valor da produção

for i, linha in enumerate(bloco_valor_producao[:8]):
    print(i, "->", linha.strip())

In [0]:
# Conferindo o início do bloco Valor da produção

for i, linha in enumerate(bloco_valor_producao[:8]):
    print(i, "->", linha.strip())

In [0]:
# Extraindo somente as linhas das UFs do bloco Valor da produção

dados_valor_producao = bloco_valor_producao[4:32]

print("Quantidade de UFs encontradas:", len(dados_valor_producao))

print("\nPrimeira UF:")
print(dados_valor_producao[0].strip())

print("\nÚltima UF:")
print(dados_valor_producao[-1].strip())

In [0]:
# Transformação do bloco: Valor da produção (Mil Reais)

registros_valor_producao = []

for linha in dados_valor_producao:
    campos = next(csv.reader([linha]))

    uf = campos[0]
    valores = campos[1:]

    for (ano, produto), valor in zip(estrutura_colunas, valores):
        registros_valor_producao.append({
            "uf": uf,
            "ano": ano,
            "produto": produto,
            "valor": valor
        })

print("Quantidade de UFs processadas:", len(dados_valor_producao))
print("Quantidade total de registros gerados:", len(registros_valor_producao))

# Exibir apenas os 10 primeiros registros
for registro in registros_valor_producao[:10]:
    print(registro)

In [0]:
# Validação da transformação: Valor da produção

contagem_valor_producao = Counter(
    registro["uf"] for registro in registros_valor_producao
)

print("Primeiro registro:")
print(registros_valor_producao[0])

print("\nÚltimo registro:")
print(registros_valor_producao[-1])

print("\nQuantidade de registros por UF:")
print(contagem_valor_producao)

print("\nTodas as UFs possuem 44 registros?",
      all(qtd == 44 for qtd in contagem_valor_producao.values()))

In [0]:
# Identificando os valores não numéricos presentes nas 5 variáveis

todos_registros = (
    registros_area_destinada
    + registros_area_colhida
    + registros_quantidade
    + registros_rendimento
    + registros_valor_producao
)

valores_nao_numericos = sorted({
    registro["valor"]
    for registro in todos_registros
    if not registro["valor"].isdigit()
})

print("Total de registros das 5 variáveis:", len(todos_registros))
print("Valores não numéricos encontrados:", valores_nao_numericos)

In [0]:
# Contagem dos símbolos especiais encontrados nos dados

from collections import Counter

contagem_simbolos = Counter(
    registro["valor"]
    for registro in todos_registros
    if registro["valor"] in ["-", "..", "..."]
)

print("Quantidade de cada símbolo especial:")

for simbolo, quantidade in contagem_simbolos.items():
    print(f"{simbolo}: {quantidade}")

print("\nTotal de valores especiais:", sum(contagem_simbolos.values()))

In [0]:
# Verificando os símbolos especiais por variável

grupos_variaveis = {
    "Área destinada à colheita": registros_area_destinada,
    "Área colhida": registros_area_colhida,
    "Quantidade produzida": registros_quantidade,
    "Rendimento médio da produção": registros_rendimento,
    "Valor da produção": registros_valor_producao
}

for nome_variavel, registros in grupos_variaveis.items():

    contagem = Counter(
        registro["valor"]
        for registro in registros
        if registro["valor"] in ["-", "..", "..."]
    )

    print(f"\n{nome_variavel}")
    print(contagem)

In [0]:
# Verificando onde aparecem os valores "..."

for nome_variavel, registros in grupos_variaveis.items():

    ocorrencias = [
        registro
        for registro in registros
        if registro["valor"] == "..."
    ]

    ufs = sorted(set(registro["uf"] for registro in ocorrencias))

    print(f"\n{nome_variavel}")
    print("Quantidade:", len(ocorrencias))
    print("UFs:", ufs)

In [0]:
# Conferindo todos os valores de Guanabara nas variáveis
# Quantidade produzida e Rendimento médio

for nome_variavel, registros in {
    "Quantidade produzida": registros_quantidade,
    "Rendimento médio da produção": registros_rendimento
}.items():

    guanabara = [
        registro
        for registro in registros
        if registro["uf"] == "Guanabara"
    ]

    print(f"\n{nome_variavel}")

    for registro in guanabara:
        if registro["valor"] != "...":
            print(registro)

In [0]:
# Verificando em quais produtos aparecem os valores ".."

for nome_variavel, registros in {
    "Quantidade produzida": registros_quantidade,
    "Rendimento médio da produção": registros_rendimento
}.items():

    ocorrencias = [
        registro
        for registro in registros
        if registro["valor"] == ".."
    ]

    produtos = Counter(
        registro["produto"]
        for registro in ocorrencias
    )

    print(f"\n{nome_variavel}")
    print("Total de '..':", len(ocorrencias))
    print("Distribuição por produto:")
    print(produtos)

## Transformação e estruturação dos dados

Após a ingestão dos dados brutos na camada Bronze, foi realizada a etapa de transformação para organizar o conteúdo do arquivo em uma estrutura adequada para análise.

O arquivo original do IBGE apresenta os dados distribuídos em cinco blocos de variáveis:

- Área destinada à colheita (Hectares)
- Área colhida (Hectares)
- Quantidade produzida (Toneladas)
- Rendimento médio da produção (Quilogramas por Hectare)
- Valor da produção (Mil Reais)

Cada bloco possui informações organizadas por Unidade da Federação, ano e produto. Durante a transformação, os cabeçalhos foram interpretados e os dados foram reorganizados para uma estrutura tabular contendo as dimensões:

- Unidade da Federação
- Ano
- Produto
- Valor

Foram identificadas 28 unidades territoriais no arquivo, incluindo o registro histórico de Guanabara, 11 anos (2015 a 2025) e quatro categorias de produto para cada ano.

Cada variável resultou em 1.232 registros (28 unidades territoriais × 11 anos × 4 produtos), totalizando 6.160 registros considerando os cinco blocos.

Também foram realizadas verificações de integridade após a transformação, confirmando que todas as unidades territoriais possuem 44 registros em cada variável.

Durante a análise dos valores foram identificados os símbolos especiais "-", ".." e "...", definidos pela própria legenda do IBGE. Esses símbolos foram mantidos temporariamente sem alteração para que seu significado seja considerado corretamente na etapa de tratamento dos dados.

As notas e a legenda presentes no arquivo original também foram preservadas como informações de referência para a interpretação e tratamento dos dados.

## Tratamento e padronização dos dados

Após a estruturação dos cinco blocos de variáveis, inicia-se o tratamento dos valores para preparação da camada Silver.

A legenda disponibilizada pelo IBGE foi utilizada como referência para interpretar os símbolos especiais presentes nos dados. Foram identificados os símbolos "-", ".." e "...", que possuem significados distintos e, portanto, não devem receber o mesmo tratamento.

Nesta etapa, os valores serão padronizados preservando também a informação original, permitindo manter a rastreabilidade entre os dados tratados na camada Silver e os dados provenientes da camada Bronze.

In [0]:
# Adicionando a identificação da variável em cada conjunto de registros

for registro in registros_area_destinada:
    registro["variavel"] = "Área destinada à colheita"

for registro in registros_area_colhida:
    registro["variavel"] = "Área colhida"

for registro in registros_quantidade:
    registro["variavel"] = "Quantidade produzida"

for registro in registros_rendimento:
    registro["variavel"] = "Rendimento médio da produção"

for registro in registros_valor_producao:
    registro["variavel"] = "Valor da produção"

# Conferência
print(registros_area_destinada[0])
print(registros_area_colhida[0])
print(registros_quantidade[0])
print(registros_rendimento[0])
print(registros_valor_producao[0])

In [0]:
# Consolidação dos cinco conjuntos de dados

registros_consolidados = (
    registros_area_destinada
    + registros_area_colhida
    + registros_quantidade
    + registros_rendimento
    + registros_valor_producao
)

print("Total de registros consolidados:", len(registros_consolidados))

print("\nPrimeiro registro:")
print(registros_consolidados[0])

print("\nÚltimo registro:")
print(registros_consolidados[-1])

In [0]:
# Tratamento dos valores conforme a legenda do IBGE

registros_tratados = []

for registro in registros_consolidados:
    
    novo_registro = registro.copy()
    valor_original = registro["valor"]

    if valor_original == "-":
        valor_tratado = 0.0

    elif valor_original in ["..", "..."]:
        valor_tratado = None

    else:
        valor_tratado = float(valor_original)

    novo_registro["valor_original"] = valor_original
    novo_registro["valor"] = valor_tratado

    registros_tratados.append(novo_registro)


print("Total de registros tratados:", len(registros_tratados))

print("\nExemplos dos primeiros registros:")
for registro in registros_tratados[:10]:
    print(registro)

In [0]:
# Validação do tratamento dos símbolos especiais

for simbolo in ["-", "..", "..."]:

    exemplos = [
        registro
        for registro in registros_tratados
        if registro["valor_original"] == simbolo
    ]

    print(f"\nSímbolo original: {simbolo}")
    print("Quantidade:", len(exemplos))
    print("Valor tratado:", exemplos[0]["valor"])
    print("Exemplo:", exemplos[0])

### Tratamento dos símbolos especiais

Durante a preparação dos dados para a camada Silver, foram identificados valores representados por símbolos especiais definidos na legenda original do IBGE.

O tratamento foi realizado respeitando o significado de cada símbolo:

- **"-"**: representa zero absoluto e foi convertido para **0**.
- **".."**: representa valor não disponível e foi convertido para **NULL**.
- **"..."**: representa valor não aplicável e foi convertido para **NULL**.

Foram identificadas 2.135 ocorrências de "-", 616 ocorrências de ".." e 198 ocorrências de "...".

Para preservar a rastreabilidade dos dados, foi mantido o campo `valor_original`, contendo o valor ou símbolo exatamente como apresentado na fonte do IBGE. O campo `valor` passou a armazenar a representação numérica tratada utilizada nas análises.

Dessa forma, valores ausentes ou não aplicáveis não são confundidos com valores iguais a zero, ao mesmo tempo em que a informação original permanece disponível para consulta.

In [0]:
# Validação da quantidade de registros após o tratamento

contagem_variaveis = Counter(
    registro["variavel"]
    for registro in registros_tratados
)

print("Total de registros:", len(registros_tratados))

print("\nQuantidade de registros por variável:")
for variavel, quantidade in contagem_variaveis.items():
    print(f"{variavel}: {quantidade}")

print(
    "\nTodas as variáveis possuem 1232 registros?",
    all(quantidade == 1232 for quantidade in contagem_variaveis.values())
)

In [0]:
# Criação do DataFrame Spark com os dados tratados

df_silver = spark.createDataFrame(registros_tratados)

print("Quantidade de registros no DataFrame:", df_silver.count())

df_silver.printSchema()

In [0]:
# Padronização dos tipos de dados da camada Silver

from pyspark.sql.functions import col

df_silver = (
    df_silver
    .withColumn("ano", col("ano").cast("integer"))
    .withColumn("valor", col("valor").cast("double"))
)

df_silver.printSchema()

print("Quantidade de registros:", df_silver.count())

In [0]:
# Visualização final dos dados tratados antes da gravação na Silver

display(
    df_silver
    .orderBy("variavel", "uf", "ano", "produto")
    .limit(20)
)

## Persistência na camada Silver

Após as etapas de estruturação, tratamento e validação, os dados estão preparados para serem persistidos na camada Silver.

A estrutura final contém os campos:

- `ano`: ano de referência do dado.
- `produto`: categoria do produto agrícola.
- `uf`: Unidade da Federação.
- `valor`: valor numérico tratado e utilizado nas análises.
- `valor_original`: valor original proveniente da fonte, preservado para rastreabilidade.
- `variavel`: indicador analisado, como área colhida, quantidade produzida ou valor da produção.

Após todas as transformações, o conjunto Silver possui 6.160 registros estruturados e padronizados. A próxima etapa consiste na persistência desses dados em uma tabela Delta no schema `workspace.silver`, permitindo consultas e análises posteriores.

In [0]:
# Persistência dos dados tratados na camada Silver

tabela_silver = "workspace.silver.ibge_pam_cafe"

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(tabela_silver)
)

print(f"Tabela Silver criada com sucesso: {tabela_silver}")

In [0]:
# Validação da tabela persistida na camada Silver

df_silver_validacao = spark.table("workspace.silver.ibge_pam_cafe")

print("Quantidade de registros na tabela Silver:")
print(df_silver_validacao.count())

print("\nSchema da tabela Silver:")
df_silver_validacao.printSchema()

In [0]:
# Validação final da distribuição dos registros na tabela Silver

display(
    df_silver_validacao
    .groupBy("variavel")
    .count()
    .orderBy("variavel")
)

## Conclusão da camada Silver

A camada Silver foi concluída após as etapas de estruturação, tratamento, padronização e validação dos dados provenientes da camada Bronze.

Os cinco indicadores presentes na fonte original do IBGE foram transformados para uma estrutura tabular padronizada:

- Área destinada à colheita
- Área colhida
- Quantidade produzida
- Rendimento médio da produção
- Valor da produção

Os símbolos especiais presentes na fonte foram tratados de acordo com a legenda disponibilizada pelo IBGE, mantendo também o valor original para garantir a rastreabilidade dos dados.

Após o processamento, foram obtidos 6.160 registros, distribuídos igualmente entre as cinco variáveis, com 1.232 registros para cada indicador.

Os dados tratados foram persistidos em formato Delta na tabela:

`workspace.silver.ibge_pam_cafe`

A validação realizada diretamente sobre a tabela persistida confirmou a quantidade esperada de registros e a integridade da estrutura final.

Com a camada Silver concluída, os dados encontram-se estruturados e preparados para as próximas etapas de análise e construção da camada Gold.